In [7]:
# Instale o Sismic caso ainda não tenha:
# pip install sismic

from sismic.io import import_from_yaml, export_to_plantuml
from sismic.interpreter import Interpreter

# =====================================
# 1) Definindo a máquina como YAML
#    no formato esperado pelo Sismic
# =====================================
machine_yaml = """
statechart:
  name: ScenarioStateMachine

  # Preamble: código Python executado na inicialização
  preamble: |
    # Aqui você pode implementar suas funções de guarda reais.
    # Por enquanto deixei sempre True para testar o fluxo.

    def given_conditions_are_true():
        # Troque pelo seu critério real de entrada (Given)
        return True

    def then_conditions_are_true():
        # Troque pelo seu critério real de sucesso (Then)
        return True

  root state:
    name: root
    initial: Idle
    states:

      - name: Idle
        transitions:
          - target: ScenarioRunning
            event: do_scenario
            guard: given_conditions_are_true()

      - name: ScenarioRunning
        transitions:
          # Transição automática (sem evento) para sucesso
          - target: Success
            guard: then_conditions_are_true()
          # Ou, caso a guarda acima seja falsa, cai em erro
          - target: Error
            guard: not then_conditions_are_true()

      - name: Success
        type: final

      - name: Error
        type: final
"""

# =====================================
# 2) Carregar a máquina a partir do YAML
# =====================================
statechart = import_from_yaml(machine_yaml)
interpreter = Interpreter(statechart)

# =====================================
# 3) Inicializar a máquina
#    (coloca no estado inicial)
# =====================================
print("Config antes da inicialização:", interpreter.configuration)
interpreter.execute_once()  # entra em root + Idle
print("Config após inicialização:", interpreter.configuration)

# =====================================
# 4) Disparar o evento do_scenario
#    e deixar o interpretador rodar
# =====================================
interpreter.queue('do_scenario')

# execute() processa todos os eventos enfileirados
# e eventless transitions até estabilizar
steps = interpreter.execute()
print("Config após do_scenario:", interpreter.configuration)

# =====================================
# 5) Verificar estado final
#    (configuration é lista de nomes de estados)
# =====================================
config = interpreter.configuration

if 'Success' in config:
    print("Cenário executado com SUCESSO!")
elif 'Error' in config:
    print("Cenário terminou com ERRO!")
else:
    print("Execução não chegou em Success ou Error. Config:", config)

# =====================================
# 6) Exportar o grafo da máquina (PlantUML)
#    Isso gera um arquivo .puml
# =====================================
plantuml_text = export_to_plantuml(statechart, filepath='scenario_state_machine.puml')
print("Arquivo PlantUML gerado: scenario_state_machine.puml")

# Agora, para gerar a imagem (fora do Python), use no terminal:
#   java -jar plantuml.jar scenario_state_machine.puml
# Isso cria um PNG (scenario_state_machine.png) com o grafo.


Config antes da inicialização: []
Config após inicialização: ['root', 'Idle']
Config após do_scenario: []
Execução não chegou em Success ou Error. Config: []
Arquivo PlantUML gerado: scenario_state_machine.puml
